# Character training data generator
This notebook generates 42x42px PNG images of characters for training data.

In [1]:
import string
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from utils import split_into_char_images

### Delete old files

In [2]:
keys = list(string.ascii_lowercase) + [str(i) for i in range(10)]
char_dict = {k: [] for k in keys}
train_data_dir = 'transformer_train_data/'
# Delete all files in the target folder
for filename in tqdm(os.listdir(train_data_dir), desc="Deleting old files"):
    file_path = os.path.join(train_data_dir, filename)
    if os.path.isfile(file_path) and filename.lower().endswith(".png"):
        os.remove(file_path)
        
os.makedirs(train_data_dir, exist_ok=True)

Deleting old files: 100%|██████████████████████████████████████████████████████| 44356/44356 [00:04<00:00, 9780.24it/s]


### Generate train data

In [3]:
import os
import cv2
import shutil
from tqdm import tqdm

# --- Config ---
original_data = "clean_train_data"
output_dir = train_data_dir  # your destination directory
os.makedirs(output_dir, exist_ok=True)

# --- Collect all PNG files ---
image_files = [f for f in os.listdir(original_data) if f.lower().endswith('.png')]
count = 0
skipped = 0

for file in tqdm(image_files, desc="Splitting CAPTCHA images"):
    # Check file name
    if not file.endswith('.png'):
        continue

    if len(file.split("-")[1].split(".png")[0]) > 1:
        skipped += 1
        continue
    
    file_path = os.path.join(original_data, file)
    label = file.split("-")[0]

    # Read image
    img = cv2.imread(file_path)
    if img is None:
        skipped += 1
        continue

    # Split into characters
    char_imgs = split_into_char_images(img)

    # Verify the split matches the number of characters in label
    if len(char_imgs) != len(label):
        skipped += 1
        continue

    # Save each character image separately
    for idx, (char_img, char_label) in enumerate(zip(char_imgs, label)):
        # Ensure consistent size
        char_img = cv2.resize(char_img, (42, 42), interpolation=cv2.INTER_LINEAR)

        # Get grayscale
        if len(char_img.shape) == 3:
            gray = cv2.cvtColor(char_img, cv2.COLOR_BGR2GRAY)
        else:
            gray = char_img

        # File name format: captcha_0_c.png
        out_name = f"{label}_{idx}_{char_label}.png"
        out_path = os.path.join(output_dir, out_name)
        cv2.imwrite(out_path, gray)
        count += 1

print(f"\n✅ Saved {count} character images to {output_dir}")
print(f"Skipped {skipped} images with incorrect length of character separation.")

Splitting CAPTCHA images: 100%|████████████████████████████████████████████████████| 8000/8000 [05:40<00:00, 23.47it/s]


✅ Saved 44363 character images to transformer_train_data/
Skipped 557 images with incorrect length of character separation.
